In [ ]:
%%configure -f
{"vCores": 4, "defaultLakehouse": {"name": "diagnostic", "id": "9d10bce5-1edc-4875-83c4-ac0a98a02775", "workspaceId": "82ad2591-974a-4ad4-ace6-e24879274a4b"}}

# fabric-rlm 0.1.11.dev0 — **EffortBanditPolicy** validation

Single-LM (gpt-5) effort climb (minimal → low → medium → high → high+parallel) on the same 7-case set the cross-model bandit run used. Targets the 4 cases that previously failed at least once: **Backprop_hard_1, DistMem_hard_1, VLIW_hard_1, MCM_hard_1**.

1 iteration only — stays under the ~60 min Azure token expiry without needing the FabricLM token-refresh fix.


In [ ]:
import sys, json, time, traceback, uuid, platform as _platform
from pathlib import Path

TIER = 'effort_bandit'
RUN_ID = time.strftime('%Y%m%d-%H%M%S') + '-' + uuid.uuid4().hex[:6]
FILES_ROOT = Path('/lakehouse/default/Files')
RUN_ROOT = FILES_ROOT / 'fabric_rlm_adaptive_validation' / TIER / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)

# Separate state.json so this doesn't collide with the cross-model bandit priors.
BANDIT_STATE_PATH = FILES_ROOT / 'fabric_rlm_adaptive_validation' / 'effort_bandit' / 'state.json'
BANDIT_STATE_PATH.parent.mkdir(parents=True, exist_ok=True)

summary = {
    'tier': TIER, 'run_id': RUN_ID, 'started_at': time.time(),
    'python': _platform.python_version(),
    'stages': [], 'iterations': [], 'passed': False, 'error': None,
}
SUMMARY_PATH = RUN_ROOT / 'summary.json'

def write_summary():
    summary['updated_at'] = time.time()
    summary['elapsed_seconds'] = summary['updated_at'] - summary['started_at']
    SUMMARY_PATH.write_text(json.dumps(summary, indent=2, default=str), encoding='utf-8')

def stage(name, **fields):
    summary['stages'].append({'stage': name, 't': time.time(), **fields})
    print(f'[stage] {name}', fields if fields else '')
    write_summary()

stage('setup', run_root=str(RUN_ROOT), bandit_state=str(BANDIT_STATE_PATH))


In [ ]:
WHEEL_PATH = '/lakehouse/default/Files/fabric_rlm_longcot/wheels/fabric_rlm-0.1.11.dev0-py3-none-any.whl'
stage('wheel_check', exists=Path(WHEEL_PATH).exists(),
      size=Path(WHEEL_PATH).stat().st_size if Path(WHEEL_PATH).exists() else 0)
import subprocess
out = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                       '--force-reinstall', '--no-deps', WHEEL_PATH],
                      capture_output=True, text=True)
stage('pip_wheel', rc=out.returncode, stderr_tail=out.stderr[-400:])
if out.returncode != 0:
    summary['error'] = 'wheel install failed'; write_summary(); raise SystemExit('wheel install failed')

try:
    import dspy
    stage('dspy_present', version=getattr(dspy, '__version__', '?'))
except ImportError:
    out2 = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'dspy>=3.1.2'],
                           capture_output=True, text=True)
    stage('pip_dspy', rc=out2.returncode, stderr_tail=out2.stderr[-400:])
    if out2.returncode != 0:
        summary['error'] = 'dspy install failed'; write_summary(); raise SystemExit('dspy install failed')
    import dspy
    stage('dspy_installed', version=getattr(dspy, '__version__', '?'))

for mod in [m for m in list(sys.modules) if m == 'fabric_rlm' or m.startswith('fabric_rlm.')]:
    sys.modules.pop(mod, None)
import fabric_rlm
from fabric_rlm.experimental import EffortBanditPolicy, EFFORT_RUNG_COST
stage('imported', version=getattr(fabric_rlm, '__version__', '?'),
      effort_cost=EFFORT_RUNG_COST)


In [ ]:
import json as _json
FIXTURE_ROOT = '/lakehouse/default/Files/fabric_rlm_adaptive_validation/fixtures'

CASES = []
with open(FIXTURE_ROOT + '/longcot_cs_hard_pilot20.jsonl') as fh:
    rows = [_json.loads(line) for line in fh]
# Same 7-case mix as the cross-model bandit run for direct comparison.
mcm = [r for r in rows if r.get('template') == 'MCM'][:3]
for r in mcm:
    CASES.append({'id': 'mcm-' + str(r['question_id']), 'question': r['prompt'],
                  'answer': r['answer'], 'template': 'MCM'})
for tmpl in ['MFMC', 'Backprop', 'DistMem', 'VLIW']:
    matches = [r for r in rows if r.get('template') == tmpl]
    if matches:
        r = matches[0]
        CASES.append({'id': tmpl.lower() + '-' + str(r['question_id']),
                      'question': r['prompt'], 'answer': r['answer'], 'template': tmpl})

# Cases that failed at least once in the cross-model bandit run
PRIOR_FAILED = {'mcm-MCM_hard_1', 'backprop-Backprop_hard_1', 'distmem-DistMem_hard_1', 'vliw-VLIW_hard_1'}
for c in CASES:
    c['prior_failed'] = c['id'] in PRIOR_FAILED
stage('cases_loaded', n=len(CASES), prior_failed=sorted(PRIOR_FAILED))


In [ ]:
import sys
sys.path.insert(0, '/lakehouse/default/Files/fabric_rlm_adaptive_validation/fixtures')
try:
    from longcot_adapter import verify_cs_response
    HAS_LONGCOT = True
except Exception as _e:
    HAS_LONGCOT = False
    print('longcot_adapter import failed:', _e)

def normalize(s):
    return ''.join((s or '').lower().split())

def make_validator(case):
    expected_answer = case.get('answer')
    if expected_answer is None:
        expected_answer = ''
    elif not isinstance(expected_answer, str):
        try:
            expected_answer = _json.dumps(expected_answer, sort_keys=True)
        except Exception:
            expected_answer = str(expected_answer)
    template = case.get('template')
    if template and HAS_LONGCOT:
        def validator(result):
            if not result.submitted or not result.payload:
                return False
            ans = result.payload.get('answer') or ''
            try:
                correct, _ = verify_cs_response(template, expected_answer, ans)
                return bool(correct)
            except Exception:
                return normalize(expected_answer) in normalize(ans)
        return validator
    norm_expected = normalize(expected_answer)
    def validator(result):
        if not result.submitted or not result.payload:
            return False
        ans = result.payload.get('answer') or ''
        return norm_expected in normalize(ans)
    return validator


In [ ]:
from fabric_rlm import RLM, FabricLM
from fabric_rlm.experimental import BanditState, EffortBanditPolicy

state = BanditState.from_path(BANDIT_STATE_PATH)
def _state_obs_total(s):
    return sum(s.total_observations(k) for k in s.priors.keys())
stage('bandit_state_loaded', total_observations=_state_obs_total(state),
      task_keys=sorted(state.priors.keys()))

# Single LM — effort climbs on the same model.
base_lm = FabricLM('gpt-5', reasoning_effort='minimal', cache=False)
stage('lm_built', base='gpt-5', start_effort='minimal')

# Single iteration to stay safely under the ~60 min Azure token expiry.
N_ITERATIONS = 1
ladder_history = []

for iter_idx in range(1, N_ITERATIONS + 1):
    iter_record = {'iter': iter_idx, 'cases': []}
    summary['iterations'].append(iter_record)
    write_summary()
    stage('iteration_start', iter=iter_idx, state_obs=_state_obs_total(state),
          state_keys=sorted(state.priors.keys()))
    for i, case in enumerate(CASES, 1):
        case_record = {'id': case['id'], 'template': case['template'],
                       'prior_failed': case.get('prior_failed', False)}
        iter_record['cases'].append(case_record)
        write_summary()
        try:
            validator = make_validator(case)
            policy = EffortBanditPolicy(
                state=state,
                task_key=case['template'],
                warmup=2,
                base_lm_spec=base_lm,
                base_reasoning_effort='minimal',
                parallel_rollouts=3,
            )
            rlm = RLM(
                signature='question -> answer',
                lm=base_lm,
                engine='adaptive',
                adaptive=dict(
                    policy=policy,
                    validator=validator,
                    max_attempts=6,
                    parallel_rollouts=1,
                ),
            )
            t0 = time.perf_counter()
            result = rlm.run({'question': case['question']})
            elapsed = time.perf_counter() - t0
            meta = (result.trajectory.metadata or {}).get('adaptive', {}) if result.trajectory else {}
            attempts = meta.get('attempts', [])
            passed_now = bool(result.submitted and validator(result))
            starting_rung = attempts[0].get('rung') if attempts else None
            for a in attempts:
                rung = a.get('rung')
                a_passed = bool(a.get('passed'))
                if rung is not None:
                    state.record(case['template'], rung, a_passed)
            case_record.update({
                'passed': passed_now, 'submitted': result.submitted,
                'elapsed_seconds': elapsed, 'starting_rung': starting_rung,
                'winner_rung': meta.get('winner_rung'),
                'stop_reason': meta.get('stop_reason'),
                'attempts': [{'rung': a.get('rung'),
                              'effort': (a.get('config') or {}).get('reasoning_effort'),
                              'parallel': (a.get('config') or {}).get('parallel_rollouts'),
                              'passed': a.get('passed')} for a in attempts],
                'n_attempts': len(attempts),
            })
            ladder_history.append({
                'iter': iter_idx, 'case_id': case['id'], 'template': case['template'],
                'starting_rung': starting_rung, 'n_attempts': len(attempts),
                'passed': passed_now, 'elapsed': elapsed,
                'prior_failed': case.get('prior_failed', False),
            })
            stage('case_done', iter=iter_idx, id=case['id'], template=case['template'],
                  starting_rung=starting_rung, n_attempts=len(attempts), passed=passed_now,
                  elapsed=round(elapsed, 1))
        except Exception as exc:
            case_record.update({'passed': False, 'error': repr(exc),
                                'traceback': traceback.format_exc()})
            stage('case_error', iter=iter_idx, id=case['id'], error=repr(exc))
        try:
            state.save()
        except Exception as save_exc:
            stage('state_save_error', error=repr(save_exc))
    passed_count = sum(1 for c in iter_record['cases'] if c.get('passed'))
    iter_record['passed_count'] = passed_count
    iter_record['total_cases'] = len(CASES)
    stage('iteration_done', iter=iter_idx, passed=passed_count, total=len(CASES))

print()
print('=== Effort-bandit ladder history ===')
print(f'{"iter":<5} {"template":<10} {"start":<6} {"att":<5} {"prior_fail":<11} {"pass":<6} {"elapsed":<8}')
for h in ladder_history:
    print(f'{h["iter"]:<5} {h["template"]:<10} {str(h["starting_rung"]):<6} {h["n_attempts"]:<5} {str(h["prior_failed"]):<11} {str(h["passed"]):<6} {h["elapsed"]:.1f}')

summary['ladder_history'] = ladder_history
summary['final_state_obs'] = _state_obs_total(state)
summary['final_state_keys'] = sorted(state.priors.keys())
summary['passed'] = True
write_summary()
print()
prior_failed_pass = sum(1 for h in ladder_history if h['prior_failed'] and h['passed'])
prior_failed_total = sum(1 for h in ladder_history if h['prior_failed'])
print(f'TIER=effort_bandit RUNS={N_ITERATIONS} STATE_OBS={_state_obs_total(state)}')
print(f'PRIOR_FAILED_NOW_PASSING={prior_failed_pass}/{prior_failed_total}')
